In [4]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt


In [5]:
data = pd.read_csv('/Users/krishvenigalla/Desktop/NutriWeb/NutriWeb/useful_data/cleaned_data.csv')
data.head()

/var/folders/fm/08v38d894qqg0x2sfcmb_2mh0000gn/T/ipykernel_29881/3635408293.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/Users/krishvenigalla/Desktop/NutriWeb/NutriWeb/useful_data/cleaned_data.csv')


,code,product_name,brands,categories_en,labels_en,ingredients_text,allergens_en,additives_en,nutrition_grade_fr,energy_100g,...,cocoa_100g,carbon-footprint_100g,nutrition-score-fr_100g,nutrition-score-uk_100g,category_level_1,category_level_2,category_level_3,category_level_4,category_level_5,category_level_6
0,4530,Banana Chips Sweetened (Whole),not mentioned,NaN,No labels,"Bananas, vegetable oil (coconut oil, corn oil ...",0,No additives,d,2243.0,...,0.0,no information,14.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN
1,4559,Peanuts,torn & glasser,NaN,No labels,"Peanuts, wheat flour, sugar, rice flour, tapio...","en:wheat, en:peanuts, en:soy",No additives,b,1941.0,...,0.0,no information,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,16087,Organic Salted Nut Mix,grizzlies,NaN,No labels,"Organic hazelnuts, organic cashews, organic wa...",0,No additives,d,2540.0,...,0.0,no information,12.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN
3,16094,Organic Polenta,bob's red mill,NaN,No labels,Organic polenta,0,No additives,not given,1552.0,...,0.0,no information,not given,not given,NaN,NaN,NaN,NaN,NaN,NaN
4,16100,Breadshop Honey Gone Nuts Granola,unfi,NaN,No labels,"Rolled oats, grape concentrate, expeller press...",en:sesame,No additives,not given,1933.0,...,0.0,no information,not given,not given,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
from modules.ingredients import clean_ingredients

data['ingredients'] = data['ingredients_text'].copy()
data['ingredients_text'] = data['ingredients_text'].apply(clean_ingredients)

### Pre-processing 

In [ ]:
# Clean and standardize text columns
def clean_text(text):
    if isinstance(text, str):
        return text.lower().strip()
    return text

data['ingredients'] = data['ingredients'].apply(clean_text)
data['category_level_1'] = data['category_level_1'].apply(clean_text)
data['category_level_2'] = data['category_level_2'].apply(clean_text)

# Replace placeholders with NaN or empty lists
data['ingredients'] = data['ingredients'].replace('ingredients are missing', np.nan)

# Split columns into lists
data['ingredients'] = data['ingredients'].str.split(', ')


### Tokenization, Stop word removal & Lemmatization

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [ ]:
from modules.ingredients import clean_ingredients
#sample_text = "Bananas, vegetable oil (coconut oil, corn oil and/or palm oil) sugar, natural banana flavor." 
sample_text = 'Skinless chicken meat, feta cheese (pasteurized milk, salt, cheese cultures, enzymes), contains 2% or less of water, sea salt, garlic, spinach flakes, dried minced onion, rosemary, black pepper, and rosemary extract.'
sample_text = clean_ingredients(sample_text)
sample_text = re.sub(r'\b(and|or|and/or)\b|[^\w\s]', '', sample_text, flags=re.IGNORECASE).strip()
tokens = word_tokenize(sample_text)
print("Tokens:", tokens)

In [ ]:
stop_words = set(stopwords.words('english'))
custom_stopwords = [
    'contains', 'containing', 'including', 'following', 'and/or', 'and', 'with', 'than', 
    'made', 'from', 'may', 'less', 'contain', 'up to', '2', '2%', '2% or less of the following',
    'or', 'of', 'the', 'a', 'an', 'in', 'on', 'at', 'for', 'to', 'by', 'is', 'are',
]
tokens2 = [word for word in tokens if word not in stop_words]
tokens2 = [word for word in tokens if word not in custom_stopwords]
print("Tokens without stop words:", tokens2)

In [ ]:
lemmatizer = WordNetLemmatizer()
tokens3 = [lemmatizer.lemmatize(word) for word in tokens2]
print(tokens3)

In [ ]:
final_text = ' '.join(tokens3)
final_text

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode the text
encoded_text = model.encode(final_text)
print("Encoded text:", encoded_text)

### Umap Visualization for products in United States

In [ ]:
from sentence_transformers import SentenceTransformer
import plotly.graph_objs as go
import plotly.express as px
import pandas as pd

df = pd.read_csv('/Users/krishvenigalla/Desktop/NutriWeb_clone/data/cleaned_data.csv')
df_filtered = df[df.countries_en == 'United States'] 
df_filtered = df_filtered.dropna(subset=['category_level_1'])
category_level_1_values = df_filtered['category_level_1'].tolist()


In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

ingredients = df_filtered['ingredients_text'].to_list()
custom_stopwords = [
    'contains', 'containing', 'including', 'following', 'and/or', 'and', 'with', 'than', 
    'made', 'from', 'may', 'less', 'contain', 'up to', '2', '2%', '2% or less of the following',
    'or', 'of', 'the', 'a', 'an', 'in', 'on', 'at', 'for', 'to', 'by', 'is', 'are']

def preprocess(text):
    
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\b(and|or|and/or)\b', '', text, flags=re.IGNORECASE).strip() 
    text = text.lower()
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [word for word in tokens if word not in custom_stopwords]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

ingredient_strings = [preprocess(item) for item in ingredients]

In [ ]:
product_names = df_filtered['product_name'].tolist()

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(ingredient_strings)

Visulaizing products from United States in a 3D plane

In [ ]:
import umap.umap_ as umap

# Step 1: Drop missing category_level_1
# df_filtered = df_sample.dropna(subset=['category_level_1'])

# Step 2: Get Top 10 frequent categories
top_10_categories = df_filtered['category_level_1'].value_counts().nlargest(10).index

# Step 3: Keep only products belonging to Top 10 categories
df_top10 = df_filtered[df_filtered['category_level_1'].isin(top_10_categories)].reset_index(drop=True)

# Step 4: Get corresponding embeddings
embeddings_top10 = embeddings[:len(df_top10)]

# Step 5: UMAP 3D projection
umap_model = umap.UMAP(n_components=3, random_state=42)
embedding_3d = umap_model.fit_transform(embeddings_top10)

# Step 6: Prepare category labels
df_top10 = df_top10.copy()
df_top10['UMAP-1'] = embedding_3d[:, 0]
df_top10['UMAP-2'] = embedding_3d[:, 1]
df_top10['UMAP-3'] = embedding_3d[:, 2]

# Step 7: Create colors for categories
unique_cats = df_top10['category_level_1'].unique()
colors = px.colors.qualitative.Set3  # Beautiful categorical colors
color_map = {cat: colors[i % len(colors)] for i, cat in enumerate(unique_cats)}

# Step 8: Create traces (one per category)
fig = go.Figure()

for cat in unique_cats:
    cat_data = df_top10[df_top10['category_level_1'] == cat]
    fig.add_trace(go.Scatter3d(
        x=cat_data['UMAP-1'],
        y=cat_data['UMAP-2'],
        z=cat_data['UMAP-3'],
        mode='markers',
        name=cat,  # Legend label
        marker=dict(
            size=5,
            color=color_map[cat],
            opacity=0.8
        ),
        text=cat_data['product_name'],  # Hover shows product name
        hoverinfo='text'
    ))

# Step 9: Update layout
fig.update_layout(
    title="Top 10 Product Categories - 3D UMAP Clustering with Legend",
    scene=dict(
        xaxis_title='UMAP-1',
        yaxis_title='UMAP-2',
        zaxis_title='UMAP-3'
    ),
    legend=dict(
        title="Product Categories",
        x=1.02,  # Move legend outside plot
        y=1
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()
fig.to_image(format="png", width=800, height=600, scale=2)
fig.write_image("3D_UMAP_Clustering.png")

Visualizing all products from United States in a 2D plane 

In [ ]:
import numpy as np
import pandas as pd


df = pd.read_csv('/Users/krishvenigalla/Desktop/NutriWeb_clone/data/cleaned_data.csv')
df_filtered_2 = df[df.countries_en == 'United States'] 
df_filtered_2 = df_filtered_2.dropna(subset=['category_level_1'])
category_level_1_values = df_filtered_2['category_level_1'].tolist()
product_names_2 = df_filtered_2['product_name'].tolist()

df_filtered_2['ingredients_text'] = df_filtered_2['ingredients_text'].replace('Ingredients are missing', np.nan)
ingredients_2 = df_filtered_2['ingredients_text'].to_list()
ingredient_strings_2 = [preprocess(item) if isinstance(item, str) else '' for item in ingredients_2]

In [ ]:
model_2 = SentenceTransformer("all-MiniLM-L6-v2")
embeddings_2 = model_2.encode(ingredient_strings_2)

In [ ]:
import umap.umap_ as umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
embedding_2d = reducer.fit_transform(embeddings_2)

df_vis = pd.DataFrame({
    'x': embedding_2d[:, 0],
    'y': embedding_2d[:, 1],
    'product_name': product_names_2,
    'ingredients': ingredient_strings_2,
    'category_level_1': category_level_1_values
})


# Step 3: Handle missing category_level_1
df_vis['category_level_1'] = df_vis['category_level_1'].fillna('Unknown') 
df_vis['category_level_1'] = df_vis['category_level_1'].replace('', 'Unknown')  

In [ ]:
import plotly.express as px

fig = px.scatter(
    df_vis, 
    x='x', 
    y='y', 
    color='category_level_1',
    hover_data=['product_name', 'ingredients'],  
    title="UMAP Projection of Food Product Ingredients",
    color_discrete_map={'Unknown': 'blue'}
)

# Update layout to set axis titles
fig.update_layout(
    xaxis_title='Ingredient Similarity – Dimension 1',
    yaxis_title='Ingredient Similarity – Dimension 2'
)

fig.show()
fig.write_html('/Users/krishvenigalla/Desktop/NutriWeb/NutriWeb/2D_UMAP_top10_categories.html')
fig.write_image("2D_UMAP_Clustering.png")

# We are trying two different type of recommendations here:

1) recommendations_code module - We use bar code of the product extensively to check for ingredients and product name of the particular item and start recommending using FAISS index. In this case the products with similar names and brands gets ignored sometimes if they do not have perfect ingredient match.

2) recommendations module - Here we follow the same methodology, but with a small tweak i.e. we give a name_weightage for the product name and measure the score using match_boost parameter. Weight to combine ingredient and name embeddings and boost to apply if product name closely matches.

### Recommendations Example Usage using bar_code

In [ ]:
from modules.recommendations_code import recommend_products
from modules.recommendations_code import recommend_by_ingredients

# Example usage
bar_code = 749826138015 
allergens_to_avoid = []  

recommendations = recommend_products(bar_code, data, top_n=5, allergens_to_avoid=allergens_to_avoid)

# Check if recommendations is None
if recommendations is None or recommendations.empty:
	print("No recommendations found. Please check the input data or function implementation.")
else:
	# Print recommendations
	print("Final recommendations:")
	print(recommendations[['product_name', 'additives_en', 'allergens_en', 'nutrition_grade_fr']].to_string(index=False))

### Recommendations Example Usage using bar_code with name weightage

In [9]:
from modules.recommendations import recommend_products 
recommend_products(bar_code=16872, df=data, top_n=5, allergens_to_avoid=['peanuts'], name_weight=0.3, match_boost=0.2)

,product_name,additives_en,allergens_en,nutrition_grade_fr
147453,Sesame Tarragon Crackers,"E322,E322i","sesame, wheat, soy",d
150741,"Terrafina, Oriental Party Mix",No additives,"wheat, sesame, soy",b
85156,"Clusters & Flakes Cereal, Vanilla Almond",E160b,"tree nuts, wheat",c
47136,"Milk Chocolate Candies, Honey Roasted Sesame S...","E102,E110,E129,E133,E1400,E150,E171,E322,E322i...","soy, sesame, wheat, milk",e
50364,"Nut-Thins, Artisan Sesame Seeds Cracker Snack",No additives,"sesame, wheat",a


In [10]:
data.ingredients_text

0         Bananas, vegetable oil (coconut oil, corn oil ...
1         Peanuts, wheat flour, sugar, rice flour, tapio...
2         Organic hazelnuts, organic cashews, organic wa...
3                                           Organic polenta
4         Rolled oats, grape concentrate, expeller press...
                                ...                        
289252                              Ingredients are missing
289253                              Ingredients are missing
289254    thé vert, arôme naturel bergamote avec autres ...
289255    Organic peppermint, organic lemon grass, organ...
289256    Citric acid, maltodextrin, instant tea, aspart...
Name: ingredients_text, Length: 289257, dtype: object

### Radial chart

In [ ]:
import pandas as pd
from modules.radar_chart import preprocess_data, create_radar_chart_with_dropdown

# Define the columns to analyze
category_col = 'category_level_1'
nutrient_cols = ['fat_100g', 'carbohydrates_100g', 'proteins_100g', 'fiber_100g', 'salt_100g']

# Preprocess the data
top_category_nutrition = preprocess_data(data, category_col, nutrient_cols, top_n=15)

# Extract categories and values
categories = top_category_nutrition[category_col]
values = top_category_nutrition[nutrient_cols]

# Create a radar chart
create_radar_chart_with_dropdown(categories, values, title='Top 15 Primary Categories by Nutritional Facts')


# Do not run the code from here

## Ingredient based recommendations using tokenization, embeddings and clustering

### Pre-processing and Vectorization

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk 
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')  
nltk.download('omw-1.4')

data['ingredients_text'] = data['ingredients_text'].replace('ingredients are missing', np.nan) 
data['ingredients_text'] = data['ingredients_text'].fillna('')
ingredients = data.ingredients_text 

### Step 1 : Text preprocessing with Tokenization, Stop word removal & Lemmatization

In [ ]:
custom_stopwords = [
    'contains', 'containing', 'including', 'following', 'and/or', 'and', 'with', 'than', 
    'made', 'from', 'may', 'less', 'contain', 'up to', '2', '2%', '2% or less of the following',
    'or', 'of', 'the', 'a', 'an', 'in', 'on', 'at', 'for', 'to', 'by', 'is', 'are',
]

def preprocess(text):
    
    text = re.sub(r'[^\w\s]', '', text)
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [word for word in tokens if word not in custom_stopwords]
    tokens = [word for word in tokens if len(word) > 1 and not word.isdigit()]
    tokens = [word for word in tokens if not re.match(r'^e\d+[a-z]*$', word)]  
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

cleaned_ingredients = [preprocess(item) for item in ingredients]

print("Cleaned Ingredients:")
cleaned_ingredients[:5]

"pasteurized free range egg.sugar.blueberries (15%).rapeseed oil.potato starch.water.cornflour.thickener.e1422.palm oil.dried whey (milk).raising agent.e450, sodium bicarbonate. emulsifier. e481, e472e, e472b, e475.flavoring.dried glucose syrup.dried skimmed milk.stabiliser. xanthan gum.salt"

In [ ]:
pd.DataFrame(cleaned_ingredients).to_csv('/Users/krishvenigalla/Desktop/cleaned_ingredients.csv', index=False)

### Step 2 : Generate Embeddings with Sentence Transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the cleaned and lemmatized data
ingredient_embeddings = model.encode(cleaned_ingredients, show_progress_bar=True)

# Print the shape of the embeddings
print(f"Embeddings shape: {ingredient_embeddings.shape}")

In [ ]:
product_name_embeddings = model.encode(data['product_name'].tolist(), show_progress_bar=True)
print(f"Product name embeddings shape: {product_name_embeddings.shape}") 

### Step 3 : Clustering with K-Means (this is a different method)

In [ ]:
# from sklearn.cluster import KMeans

# k = 10  
# kmeans = KMeans(n_clusters=k, random_state=42)
# cluster_labels = kmeans.fit_predict(ingredient_embeddings)

# # Map cluster labels back to ingredients
# clustered_ingredients = pd.DataFrame({
#     'Ingredient': ingredients,
#     'Cleaned_Ingredient': cleaned_ingredients,
#     'Cluster': cluster_labels
# })


# print("\nClustered Data:")
# print(clustered_ingredients)

### Step 3 : Similarity search with FAISS

In [ ]:
import faiss
import numpy as np

# Step 1: Normalize embeddings (critical for cosine similarity)
ingredient_embeddings_normalized = ingredient_embeddings / np.linalg.norm(ingredient_embeddings, axis=1, keepdims=True)

# Step 2: Create FAISS index (cosine similarity = Inner Product on normalized vectors)
dimension = ingredient_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product ≈ Cosine Similarity
index.add(ingredient_embeddings_normalized.astype('float32'))

# Step 3: Query (normalize the query too!)
query = "dark chocolate bar"
query_embedding = model.encode([query])
query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding, axis=1, keepdims=True)
k = 5
distances, indices = index.search(query_embedding_normalized.astype('float32'), k)

# Step 4: Map results to ingredients
results = [(ingredients[idx], distances[0][i]) for i, idx in enumerate(indices[0])]
print(f"Top {k} matches for '{query}': {results}")

### Save the embeddings

In [ ]:
import numpy as np

# Save embeddings to a file
np.save('/Users/krishvenigalla/Desktop/NutriWeb_clone/embeddings/ingredient_embeddings.npy', ingredient_embeddings)
np.save('/Users/krishvenigalla/Desktop/NutriWeb_clone/embeddings/product_name_embeddings.npy', product_name_embeddings) 